# M6b 랜덤포레스트 · 앙상블 — 실습 (W10, M6 2부작 완결편)

> ⚠️ **가장 먼저 — 화면 위 [Drive로 복사]를 누르세요.**
> 지금 보고 있는 것은 원본을 잠깐 띄운 **임시 사본**입니다. 복사하지 않으면 탭을 닫는 순간
> 채운 빈칸과 실행 결과가 **모두 사라집니다.** 복사본은 내 Google Drive에 저장되고,
> 원본은 바뀌지 않으니 마음껏 고쳐도 됩니다.

> 위에서부터 한 셀씩 `Shift+Enter`로 실행하세요. `___` 빈칸은 직접 채웁니다.

**이 실습이 끝나면**
1. **다수결의 수학**(70% 나무 3그루 → 0.784)을 손계산·시뮬레이션으로 재현하고, **마법이 깨지는 반례**(복제 0.697, 40% → 악화)를 확인한다 ⭐
2. 부트스트랩의 "**37%가 빠진다**"(0.3664)를 손계산·시뮬레이션으로 검증한다
3. **미니 숲을 손수 지어**(개별 평균 0.921 → 투표 1.0), `RandomForestClassifier`(수렴·안정·OOB)와 **특징 중요도**(M6a 루트 경합과 재회)를 다룬다

**7단계 멘탈모델 초점:** 모델(Model) — 모델들의 조합(앙상블)

## Part A. 세트피스 — 다수결의 수학 ⭐
정확도 70% 나무 3그루가 **서로 독립적으로** 틀릴 때, 다수결(2표 이상)의 정확도는? 먼저 종이에서: 셋 다 맞음 0.7³ = 0.343, 둘만 맞음 3 × 0.7² × 0.3 = 0.441 → 합 **0.784**. 코드로 재현하고 10만 문제 시뮬레이션으로 검증합니다.

In [ ]:
import numpy as np                                     # 수치 계산
import matplotlib.pyplot as plt                        # 그래프

p = 0.7                                                # 나무 한 그루의 정확도
acc3 = 3 * p**2 * (1 - p) + ___                        # ✍️ 빈칸: 둘만 맞음 + 셋 다 맞음(0.7의 세제곱)
print('손계산 — 3그루 다수결:', round(acc3, 4))          # 0.784

rng = np.random.default_rng(0)                         # 재현성
N = 100000                                             # 문제 10만 개
correct = rng.random((3, N)) < p                       # 나무 3그루가 각자 독립적으로 맞힘/틀림
votes = correct.sum(axis=0)                            # 문제별 '맞힌 나무 수'(0~3표)
print('시뮬레이션 — 독립 3그루:', round((votes >= ___).mean(), 4))  # ✍️ 빈칸: 다수결 기준(몇 표 이상?)

clone = rng.random(N) < p                              # 복제 나무: 셋이 항상 같이 맞고 같이 틀림
print('시뮬레이션 — 복제 3그루:', round(clone.mean(), 4))  # 0.697 — 투표 이득 0!

In [ ]:
from math import comb                                  # 조합(경우의 수)

def majority_acc(p, k):                                # k그루(홀수) 다수결 정확도
    need = k // 2 + 1                                  # 과반 표
    return sum(comb(k, m) * p**m * (1 - p)**(k - m) for m in range(need, k + 1))  # 과반 이상 맞을 확률

ks = range(1, 26, 2)                                   # 나무 1~25그루(홀수)
for pv, style in [(0.7, 'o-'), (0.5, 's--'), (0.4, 'v:')]:   # 70% / 동전 / 40%
    plt.plot(list(ks), [majority_acc(pv, k) for k in ks], style, label=f'p = {pv}')
plt.xlabel('number of trees (odd)'); plt.ylabel('majority-vote accuracy')  # 축(영어)
plt.legend(); plt.grid(True, alpha=0.3)
plt.title('Majority vote: better-than-chance helps, worse hurts')
plt.show()
print('p=0.7:', [round(majority_acc(0.7, k), 3) for k in (1, 3, 5, 21, 101)])  # 0.7→0.784→0.837→0.974→~1.0
print('p=0.4, 3그루:', round(majority_acc(0.4, 3), 3))  # 0.352 — 모을수록 악화!

> **검산 포인트:** 손계산 0.784 = 시뮬레이션 0.783(10만 문제). **복제 나무는 0.697** — 같이 틀리면 투표 이득이 정확히 0. 그래프: p=0.7은 나무를 늘릴수록 1.0으로, p=0.5는 평평하게, **p=0.4는 0으로** 떨어집니다. **조건: ① 반타작 이상 ② 서로 다른 실수.** 남은 문제 — 같은 train에서 어떻게 "서로 다른" 나무를 만드나?

## Part B. 배깅의 산수 — 37%가 빠진다
복원추출(bootstrap)에서 특정 샘플이 **한 번도 안 뽑힐 확률** = (1 − 1/n)ⁿ. 손계산과 시뮬레이션을 대조하세요.

In [ ]:
n = 124                                                # train 크기(와인 70% — M6a와 동일)
p_never = (1 - 1 / n) ** ___                           # ✍️ 빈칸: n번 뽑는 동안 계속 피해갈 확률(지수는?)
print('손계산 (123/124)^124 =', round(p_never, 4))       # 0.3664 (~ 1/e = 0.3679)

rng = np.random.default_rng(0)                         # 재현성
miss = []                                              # 미포함 비율 저장
for _ in range(1000):                                  # 부트스트랩 1,000벌
    idx = rng.integers(0, n, n)                        # 복원추출 n개
    miss.append(1 - len(np.unique(idx)) / n)           # 안 뽑힌 비율
print('시뮬레이션 평균 미포함 비율:', round(np.mean(miss), 4))  # 0.366 — 손계산과 일치

> **관찰:** 놀랍게도 **약 37%가 빠집니다**(n이 커지면 1/e ≈ 0.368로 수렴). 나무마다 "본 데이터"가 이만큼 다르니 트리도 달라질 수밖에 — M6a Part E에서 루트부터 갈렸던 이유. 이 빠진 37%는 Part D에서 **공짜 검증 세트(OOB)** 로 재활용됩니다.

## Part C. 미니 숲을 손수 짓기
라이브러리 없이 배깅을 직접 구현합니다: 부트스트랩 15벌 × 깊이 무제한 트리 → **다수결**. 개별 나무와 숲의 점수를 비교하세요.

In [ ]:
from sklearn.datasets import load_wine                 # 와인 데이터(M6a와 동일)
from sklearn.model_selection import train_test_split   # 공정한 시험(M2a)
from sklearn.tree import DecisionTreeClassifier        # 나무 재료

wine = load_wine()                                     # 178병, 13특징, 3품종
X, y = wine.data, wine.target
X_train, X_test, y_train, y_test = train_test_split(   # M6a와 같은 분할
    X, y, test_size=0.3, random_state=42, stratify=y)

ntr = len(X_train)                                     # 124
rng = np.random.default_rng(42)                        # 재현성
preds = []                                             # 나무별 예측 저장
for i in range(15):                                    # 나무 15그루
    idx = rng.integers(0, ntr, ntr)                    # 부트스트랩(복원추출) — Part B의 그것
    t = DecisionTreeClassifier(random_state=0).fit(X_train[idx], y_train[idx])  # 깊이 무제한(일부러!)
    preds.append(t.predict(X_test))                    # 시험 예측
preds = np.array(preds)                                # (15, 54)

indiv = [(pr == y_test).mean() for pr in preds]        # 개별 나무 점수들
print('개별 나무: 평균', round(np.mean(indiv), 3), '| 최저', round(min(indiv), 3), '| 최고', round(max(indiv), 3))

maj = np.array([np.bincount(col).___() for col in preds.T])  # ✍️ 빈칸: 표가 가장 많은 품종(다수결)
print('미니 숲(15그루 다수결):', round((maj == y_test).mean(), 3))  # 1.0!
single = DecisionTreeClassifier(random_state=0).fit(X_train, y_train)  # 비교용 단일 트리
print('단일 트리(깊이 무제한):', round(single.score(X_test, y_test), 3))  # 0.963

> **관찰:** 개별 나무는 평균 **0.921**(최저 0.778짜리도 있음) — 그런데 15그루의 다수결은 **1.0**. Part A의 수학이 실제 데이터에서 작동하는 순간입니다. 깊이 무제한(과적합 나무!)을 일부러 쓴 이유: 배깅은 **불안정하고 다양한 나무**일수록 투표 이득이 큽니다.

## Part D. 진짜 숲 — RandomForestClassifier
sklearn의 랜덤포레스트(배깅 + **특징 무작위**까지)로 수렴·안정성·OOB를 확인합니다.

In [ ]:
from sklearn.ensemble import RandomForestClassifier    # 랜덤포레스트

for ne in (1, 10, 50, 200):                            # 나무 수를 늘려가며
    r = RandomForestClassifier(n_estimators=ne, random_state=0).fit(X_train, y_train)
    print(f'n_estimators={ne:>3}:', round(r.score(X_test, y_test), 3))  # 0.889 → 0.981 → 1.0 → 1.0

rf = RandomForestClassifier(n_estimators=___, random_state=0).fit(X_train, y_train)  # ✍️ 빈칸: 수렴한 나무 수(위 실험의 최댓값)
print('최종 숲:', round(rf.score(X_test, y_test), 3))    # 1.0

for s in (0, 1, 2):                                    # 시드를 바꿔도?(M6a 불안정성과 대비)
    r = RandomForestClassifier(n_estimators=200, random_state=s).fit(X_train, y_train)
    print('seed', s, ':', round(r.score(X_test, y_test), 3))  # 전부 1.0 — 숲은 흔들리지 않는다

rf_oob = RandomForestClassifier(n_estimators=200, random_state=0,
                                oob_score=___).fit(X_train, y_train)  # ✍️ 빈칸: OOB 채점 켜기
print('OOB score:', round(rf_oob.oob_score_, 3))        # 0.968 — 빠진 37%로 공짜 검증

> **관찰:** 나무 수는 **수렴**(50에서 이미 1.0 — 무한정 늘릴 필요 없음). 시드 0/1/2 전부 1.0 — 지난주 "루트부터 갈리던" 불안정의 **구조적 해결**. OOB 0.968은 test 없이 얻은 일반화 추정(최종 보고는 그래도 봉인된 test로 — M2a). ⚠️ 1.0은 와인이 작고 깨끗해서(test 54병) — 실전에서 1.0이면 **누수부터 의심**(M2a).

## Part E. 특징 중요도 — 지난주와의 재회
숲의 보너스: 각 특징이 불순도(M6a의 지니!)를 얼마나 줄였는지 합산한 **특징 중요도**. 지난주 루트 경합의 그 특징들이 다시 나올까요?

In [ ]:
importances = rf.___                                   # ✍️ 빈칸: 특징 중요도 속성(합계 1.0)
order = np.argsort(importances)[::-1]                  # 높은 순 정렬

plt.figure(figsize=(10, 4.5))
plt.bar(range(len(importances)), importances[order])   # 중요도 막대
plt.xticks(range(len(importances)), np.array(wine.feature_names)[order], rotation=90)
plt.ylabel('importance'); plt.title('Feature importances (RF, 200 trees)')  # 축(영어)
plt.tight_layout(); plt.show()

for i in order[:4]:                                    # 상위 4개
    print(f'{wine.feature_names[i]:>16}: {importances[i]:.3f}')
print('합계:', round(importances.sum(), 3))             # 1.0

> **재회:** 상위 4 = **color_intensity(0.172) · proline(0.168) · flavanoids(0.153) · alcohol(0.127)** — 지난주 M6a 불안정성 실험에서 **루트를 두고 경합하던 바로 그 4개**(시드 12개의 루트 분포: color_intensity 5회 · alcohol 4회 · proline 2회 · flavanoids 1회). 단일 트리의 불안한 경합이 숲에서는 **집계된 순위표**가 됩니다. ⚠️ 중요도 ≠ 인과, 기본 방식은 편향 가능(순열 중요도로 교차확인 — 🔹심화).

## 🤖 AI 코파일럿 활용 (선택) — ai-native v1
막히면 AI 튜터에게 묻되, **먼저 스스로 생각**하고 답을 **실행으로 검증**하세요.

**좋은 질문 예시**
- "80% 나무 3그루의 다수결 정확도를 내가 계산해 볼 테니 채점해 줘."
- "복제 나무 100그루가 소용없는 이유를 '같이 틀린다'로 설명해 볼게 — 허점을 찔러 줘."
- "배깅의 다양성과 특징 무작위의 다양성이 어떻게 다른지 말해 볼게."
- "max_features=None(특징 무작위 끄기)으로 하면 숲이 어떻게 변할지 예측해 볼게 — 실행으로 검증할게."

**가드레일**
1. 먼저 손으로 생각 → 그 다음 AI
2. AI 코드는 *왜 그런지* 설명할 수 있을 때만 사용
3. AI 출력은 실행으로 검증

## 정리 & 자가 점검

**오늘 한 일 3줄**
1. **다수결의 수학**을 손계산(0.784)·시뮬레이션(0.783)으로 재현하고, 반례(복제 0.697 · p=0.4는 악화)로 **조건 2가지**를 확인했다
2. 부트스트랩의 "37%가 빠진다"(0.3664 = 시뮬레이션 0.366)를 검증하고, **미니 숲**(개별 평균 0.921 → 투표 1.0)을 손수 지었다
3. `RandomForestClassifier`로 수렴(50그루에서 1.0)·안정성(시드 불변)·OOB(0.968)·**특징 중요도**(M6a 루트 경합 4인방과 재회)를 확인했다

**스스로 점검**
- [ ] 70% 나무 3그루의 다수결(0.343 + 0.441 = 0.784)을 손으로 쓸 수 있다
- [ ] 투표의 조건 2가지와 깨지는 반례 2가지를 말할 수 있다
- [ ] (1 − 1/n)ⁿ ≈ 37%가 무엇의 확률인지, 어디에 재활용되는지(OOB) 안다
- [ ] 배깅(데이터 다양성)과 특징 무작위(질문 다양성)를 구분한다
- [ ] 특징 중요도의 주의사항 2가지(인과 아님, 편향)를 안다

**🔹심화 (선택)**
- `max_features=None`으로 특징 무작위를 끄고 숲을 다시 지어 보세요 — 점수·중요도 분포가 어떻게 변하나요?
- `sklearn.inspection.permutation_importance`로 **순열 중요도**를 구해 기본 중요도와 순위를 비교해 보세요.
- `GradientBoostingClassifier`로 같은 데이터를 학습해 보세요 — "순차 보완" 유파의 맛보기.

**다음 시간(M7):** 세 번째 경계 — 마진을 최대로 하는 SVM, 그리고 커널.